# 04 · Evaluate on the ESCAPE benchmark test split

Loads the model trained in notebook 03 (`models/amp_bert_escape/`) and evaluates it
on the ESCAPE test split.

> **Setup:** place `escape_test.csv` under `data/raw/escape/`.

In [ ]:
# --- bootstrap: works locally and on Google Colab ---
import sys, pathlib

REPO = "https://github.com/BioGavin/AMP-BERT.git"
try:
    import amp_bert  # already installed / on sys.path
except ModuleNotFoundError:
    root = pathlib.Path.cwd()
    root = root.parent if root.name == 'notebooks' else root
    src = root / 'src'
    if not (src / 'amp_bert').exists():
        # fresh Colab runtime: clone the repo
        import subprocess
        subprocess.run(['git', 'clone', '--depth', '1', REPO], check=True)
        src = pathlib.Path('AMP-BERT') / 'src'
    sys.path.insert(0, str(src))

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
from transformers import Trainer, AutoModelForSequenceClassification
from amp_bert import AmpDataset, load_dataset, compute_metrics, build_training_args
from amp_bert.config import RAW_DIR, MODELS_DIR, RESULTS_DIR

In [ ]:
ESCAPE_TEST_CSV = RAW_DIR / 'escape' / 'escape_test.csv'
assert ESCAPE_TEST_CSV.exists(), f'ESCAPE test data not found at {ESCAPE_TEST_CSV}.'

## Load test split

In [ ]:
test_df = load_dataset(ESCAPE_TEST_CSV)
print(test_df.shape)
test_dataset = AmpDataset(test_df, max_len=200)

## Load model & predict

In [ ]:
model_path = MODELS_DIR / 'amp_bert_escape'
assert model_path.exists(), f'No model at {model_path}. Run 03_escape_train.ipynb first.'
model = AutoModelForSequenceClassification.from_pretrained(str(model_path))
eval_args = build_training_args(str(RESULTS_DIR / 'escape_test'), do_train=False,
                                per_device_eval_batch_size=56)
trainer = Trainer(model=model, args=eval_args, compute_metrics=compute_metrics)

In [ ]:
predictions, label_ids, metrics = trainer.predict(test_dataset)
metrics

In [ ]:
test_df = test_df.copy()
test_df['pred'] = predictions.argmax(-1)
out_csv = RESULTS_DIR / 'escape_test_predictions.csv'
test_df.to_csv(out_csv)
print('wrote', out_csv)